# YOLOv7 Volleyball Detection

**Pipeline:** Clone Repo → Dataset Download → Training → Validation → Testing → CSV Metrics Export

**Classes:**
- 0: ball
- 1: player
- 2: player 1
- 3: referee

**Note:** YOLOv7 uses the WongKinYiu/yolov7 repository, not ultralytics.

## 1. Setup & Install Dependencies

In [1]:
!pip install roboflow torch torchvision tqdm matplotlib scipy pyyaml tensorboard seaborn pandas requests

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 205.0 MB/s  0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.4.3
    Uninstalling numpy-2.4.3:
      Successfully uninstalled numpy-2.4.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [seaborn]m1/2 [seaborn]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.


In [2]:
import os
import sys
import csv
import re
import json
import logging
import subprocess
from datetime import datetime

import torch
from roboflow import Roboflow

print(f"Python      : {sys.version}")
print(f"PyTorch     : {torch.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU         : {torch.cuda.get_device_name(0)}")
    print(f"VRAM        : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

Python      : 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 13:09:17) [GCC 11.2.0]
PyTorch     : 2.5.1+cu121
CUDA avail  : True
GPU         : Tesla T4
VRAM        : 14.6 GB


## 2. Clone YOLOv7 Repository

In [3]:
SCRIPT_DIR = os.path.abspath(os.getcwd())
YOLOV7_DIR = os.path.join(SCRIPT_DIR, "yolov7")

if not os.path.isdir(YOLOV7_DIR):
    print("📥 Cloning YOLOv7 repository…")
    !git clone https://github.com/WongKinYiu/yolov7.git "{YOLOV7_DIR}"
else:
    print(f"✅ YOLOv7 repo already exists at: {YOLOV7_DIR}")

# Install YOLOv7 requirements
req_file = os.path.join(YOLOV7_DIR, "requirements.txt")
if os.path.isfile(req_file):
    !pip install -r "{req_file}" -q
    print("✅ YOLOv7 requirements installed")

print(f"YOLOV7_DIR: {YOLOV7_DIR}")

📥 Cloning YOLOv7 repository…
Cloning into '/teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7'...

  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      Traceback (most recent call last):
        File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 389, in <module>
          main()
        File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 373, in main
          json_out["return_val"] = hook(**hook_input["kwargs"])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "/home/zeus/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pip/_vendor/pyproject_hooks/_in_process/_in_process.py", line 137, in get_requires_for_build_wheel
          backend = _build_backend()
                    ^^^^

## 3. Download Dataset from Roboflow

In [4]:
rf = Roboflow(api_key="MUEDCloFvcri2QqbuoB0")
project = rf.workspace("jeevaraj-s").project("volleyball-hwxp2-wq92v")
version = project.version(2)
dataset = version.download("yolov7")

print(f"\n✅ Dataset downloaded to: {dataset.location}")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to volleyball-2 in yolov7pytorch:: 100%|██████████| 314/314 [00:00<00:00, 3750.43it/s]


✅ Dataset downloaded to: /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2


## 4. Configure Paths

In [5]:
# ────────────────────────── CONFIGURATION ──────────────────────────
BASE_DIR    = os.path.dirname(SCRIPT_DIR)
DATASET_DIR = os.path.abspath(dataset.location)
DATA_YAML   = os.path.join(DATASET_DIR, "data.yaml")

# YOLOv7 pretrained weights (tiny for faster training)
WEIGHTS_URL  = "https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt"
WEIGHTS_PATH = os.path.join(YOLOV7_DIR, "yolov7-tiny.pt")

# Training hyper-params
EPOCHS      = 100
IMG_SIZE    = 640
BATCH_SIZE  = 4

# Output directories
PROJECT_DIR = os.path.join(SCRIPT_DIR, "runs")
LOG_DIR     = os.path.join(SCRIPT_DIR, "logs")
RUN_NAME    = "volleyball_train"

print(f"SCRIPT_DIR  : {SCRIPT_DIR}")
print(f"YOLOV7_DIR  : {YOLOV7_DIR}")
print(f"DATASET_DIR : {DATASET_DIR}")
print(f"DATA_YAML   : {DATA_YAML}")
print(f"WEIGHTS_PATH: {WEIGHTS_PATH}")
print(f"PROJECT_DIR : {PROJECT_DIR}")

SCRIPT_DIR  : /teamspace/studios/this_studio/Volleyball01/YOLOv7
YOLOV7_DIR  : /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7
DATASET_DIR : /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2
DATA_YAML   : /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/data.yaml
WEIGHTS_PATH: /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/yolov7-tiny.pt
PROJECT_DIR : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs


In [6]:
# Download pretrained weights if not present
if not os.path.isfile(WEIGHTS_PATH):
    print(f"📥 Downloading YOLOv7-tiny weights…")
    import urllib.request
    urllib.request.urlretrieve(WEIGHTS_URL, WEIGHTS_PATH)
    print(f"✅ Weights saved to: {WEIGHTS_PATH}")
else:
    print(f"✅ Weights already exist: {WEIGHTS_PATH}")

📥 Downloading YOLOv7-tiny weights…
✅ Weights saved to: /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/yolov7-tiny.pt


In [7]:
# Update data.yaml with absolute paths
import yaml

with open(DATA_YAML, 'r', encoding='utf-8') as f:
    data_config = yaml.safe_load(f)

# Set absolute paths for train/val/test
data_config['train'] = os.path.join(DATASET_DIR, 'train', 'images')
data_config['val']   = os.path.join(DATASET_DIR, 'valid', 'images')
data_config['test']  = os.path.join(DATASET_DIR, 'test', 'images')

with open(DATA_YAML, 'w', encoding='utf-8') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ Updated data.yaml with absolute paths:")
print(f"   train: {data_config['train']}")
print(f"   val  : {data_config['val']}")
print(f"   test : {data_config['test']}")
print(f"   nc   : {data_config['nc']}")
print(f"   names: {data_config['names']}")

✅ Updated data.yaml with absolute paths:
   train: /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/train/images
   val  : /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/valid/images
   test : /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/test/images
   nc   : 4
   names: ['ball', 'player', 'player 1', 'referee']


## 5. GPU Check

In [8]:
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

if torch.cuda.is_available():
    DEVICE = "0"
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"✅ GPU detected: {name}  ({mem:.1f} GB VRAM)")
else:
    DEVICE = "cpu"
    print("❌ No GPU found — falling back to CPU (training will be slow).")

print(f"Using device: {DEVICE}")

✅ GPU detected: Tesla T4  (14.6 GB VRAM)
Using device: 0


## 6. Setup Logger

In [9]:
os.makedirs(LOG_DIR, exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
LOG_FILE  = os.path.join(LOG_DIR, f"training_{timestamp}.log")

logger = logging.getLogger("yolov7_train")
logger.setLevel(logging.INFO)
logger.handlers.clear()

ch = logging.StreamHandler(sys.stdout)
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter("%(message)s"))
logger.addHandler(ch)

fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
fh.setLevel(logging.INFO)
fh.setFormatter(logging.Formatter("%(asctime)s | %(message)s", datefmt="%Y-%m-%d %H:%M:%S"))
logger.addHandler(fh)

logger.info(f"Log file → {LOG_FILE}")

Log file → /teamspace/studios/this_studio/Volleyball01/YOLOv7/logs/training_20260329_100730.log


## 7. Train YOLOv7

In [10]:
TRAIN_OUTPUT_DIR = os.path.join(PROJECT_DIR, RUN_NAME)

logger.info("=" * 60)
logger.info("  YOLOv7 Volleyball Detection — GPU Training")
logger.info("=" * 60)
logger.info(f"  Weights    : {WEIGHTS_PATH}")
logger.info(f"  Dataset    : {DATA_YAML}")
logger.info(f"  Epochs     : {EPOCHS}")
logger.info(f"  Image size : {IMG_SIZE}")
logger.info(f"  Batch size : {BATCH_SIZE}")
logger.info(f"  Output     : {TRAIN_OUTPUT_DIR}")
logger.info("=" * 60)

train_cmd = [
    sys.executable, os.path.join(YOLOV7_DIR, "train.py"),
    "--workers", "0",
    "--device", DEVICE,
    "--batch-size", str(BATCH_SIZE),
    "--data", DATA_YAML,
    "--img", str(IMG_SIZE), str(IMG_SIZE),
    "--cfg", os.path.join(YOLOV7_DIR, "cfg", "training", "yolov7-tiny.yaml"),
    "--weights", WEIGHTS_PATH,
    "--name", RUN_NAME,
    "--project", PROJECT_DIR,
    "--epochs", str(EPOCHS),
    "--exist-ok",
    "--hyp", os.path.join(YOLOV7_DIR, "data", "hyp.scratch.tiny.yaml"),
]

logger.info("Starting training…\n")
logger.info(f"Command: {' '.join(train_cmd)}\n")

process = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=YOLOV7_DIR,
)

for line in process.stdout:
    line = line.rstrip()
    print(line)
    logger.handlers[1].stream.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} | {line}\n")
    logger.handlers[1].stream.flush()

process.wait()
print(f"\n{'='*60}")
print(f"Training finished with exit code: {process.returncode}")
print(f"{'='*60}")

  YOLOv7 Volleyball Detection — GPU Training
  Weights    : /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/yolov7-tiny.pt
  Dataset    : /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/data.yaml
  Epochs     : 100
  Image size : 640
  Batch size : 4
  Output     : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train
Starting training…

Command: /home/zeus/miniconda3/envs/cloudspace/bin/python /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/train.py --workers 0 --device 0 --batch-size 4 --data /teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/data.yaml --img 640 640 --cfg /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/cfg/training/yolov7-tiny.yaml --weights /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/yolov7-tiny.pt --name volleyball_train --project /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs --epochs 100 --exist-ok --hyp /teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/da

In [11]:
# ── Print Training Summary ──
BEST_WEIGHTS = os.path.join(TRAIN_OUTPUT_DIR, "weights", "best.pt")
LAST_WEIGHTS = os.path.join(TRAIN_OUTPUT_DIR, "weights", "last.pt")

logger.info("\n" + "=" * 60)
logger.info("  TRAINING COMPLETE")
logger.info("=" * 60)
logger.info(f"📁 Results saved to : {TRAIN_OUTPUT_DIR}")
logger.info(f"🏆 Best weights     : {BEST_WEIGHTS} {'✅' if os.path.isfile(BEST_WEIGHTS) else '❌'}")
logger.info(f"   Last weights     : {LAST_WEIGHTS} {'✅' if os.path.isfile(LAST_WEIGHTS) else '❌'}")

# Check for generated charts
logger.info("📊 Auto-saved charts:")
for chart in ["results.png", "confusion_matrix.png", "F1_curve.png",
              "P_curve.png", "R_curve.png", "PR_curve.png"]:
    path = os.path.join(TRAIN_OUTPUT_DIR, chart)
    status = "✅" if os.path.isfile(path) else "—"
    logger.info(f"   {status} {chart}")
logger.info("=" * 60)


  TRAINING COMPLETE
📁 Results saved to : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train
🏆 Best weights     : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt ✅
   Last weights     : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/last.pt ✅
📊 Auto-saved charts:
   ✅ results.png
   ✅ confusion_matrix.png
   ✅ F1_curve.png
   ✅ P_curve.png
   ✅ R_curve.png
   ✅ PR_curve.png


## 8. Validate YOLOv7

In [12]:
VAL_OUTPUT_DIR = os.path.join(SCRIPT_DIR, 'runs', 'val', 'volleyball_val')

if not os.path.isfile(BEST_WEIGHTS):
    print(f'❌ Error: Best weights not found at {BEST_WEIGHTS}')
else:
    print(f'✅ Loading best model: {BEST_WEIGHTS}')
    print(f'🚀 Starting Validation…')

    val_cmd = [
        sys.executable, os.path.join(YOLOV7_DIR, 'test.py'),
        '--data', DATA_YAML,
        '--img', str(IMG_SIZE),
        '--batch-size', str(BATCH_SIZE),
        '--weights', BEST_WEIGHTS,
        '--device', DEVICE,
        '--name', 'volleyball_val',
        '--project', os.path.join(SCRIPT_DIR, 'runs', 'val'),
        '--exist-ok',
        '--task', 'val',
        '--save-json',
        '--save-txt',
        '--save-conf',
        '--verbose',
    ]

    val_process = subprocess.Popen(
        val_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=YOLOV7_DIR,
    )

    val_output = []
    for line in val_process.stdout:
        line = line.rstrip()
        print(line)
        val_output.append(line)

    val_process.wait()

    # Parse validation metrics from output
    val_p = val_r = val_map50 = val_map95 = 0.0
    for line in val_output:
        parts = line.split()
        if len(parts) > 0 and parts[0] == 'all':
            try:
                float_vals = []
                for p in parts:
                    try:
                        float_vals.append(float(p))
                    except ValueError:
                        continue
                if len(float_vals) >= 4:
                    # YOLOv7 output: P R mAP50 mAP50-95 are the last 4 float columns
                    val_p     = float_vals[-4]
                    val_r     = float_vals[-3]
                    val_map50 = float_vals[-2]
                    val_map95 = float_vals[-1]
            except Exception:
                pass

    print("\n" + "━" * 50)
    print("📈 YOLOv7 VALIDATION METRICS")
    print("━" * 50)
    print(f"Precision:   {val_p:.4f}")
    print(f"Recall:      {val_r:.4f}")
    print(f"mAP@50:      {val_map50:.4f}")
    print(f"mAP@50-95:   {val_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {VAL_OUTPUT_DIR}")


✅ Loading best model: /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt
🚀 Starting Validation…
Namespace(weights=['/teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt'], data='/teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/data.yaml', batch_size=4, img_size=640, conf_thres=0.001, iou_thres=0.65, task='val', device='0', single_cls=False, augment=False, verbose=True, save_txt=True, save_hybrid=False, save_conf=True, save_json=True, project='/teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/val', name='volleyball_val', exist_ok=True, no_trace=False, v5_metric=False)
YOLOR 🚀 v0.1-128-ga207844 torch 2.5.1+cu121 CUDA:0 (Tesla T4, 14911.6875MB)

/teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/models/experimental.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to cons

## 9. Test YOLOv7

In [13]:
TEST_OUTPUT_DIR = os.path.join(SCRIPT_DIR, 'runs', 'val', 'volleyball_test')

if not os.path.isfile(BEST_WEIGHTS):
    print(f'❌ Error: Best weights not found at {BEST_WEIGHTS}')
else:
    print(f'✅ Loading best model: {BEST_WEIGHTS}')
    print(f'🚀 Starting Testing…')

    test_cmd = [
        sys.executable, os.path.join(YOLOV7_DIR, 'test.py'),
        '--data', DATA_YAML,
        '--img', str(IMG_SIZE),
        '--batch-size', str(BATCH_SIZE),
        '--weights', BEST_WEIGHTS,
        '--device', DEVICE,
        '--name', 'volleyball_test',
        '--project', os.path.join(SCRIPT_DIR, 'runs', 'val'),
        '--exist-ok',
        '--task', 'test',
        '--save-json',
        '--save-txt',
        '--save-conf',
        '--verbose',
    ]

    test_process = subprocess.Popen(
        test_cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        cwd=YOLOV7_DIR,
    )

    test_output = []
    for line in test_process.stdout:
        line = line.rstrip()
        print(line)
        test_output.append(line)

    test_process.wait()

    # Parse test metrics from output
    test_p = test_r = test_map50 = test_map95 = 0.0
    for line in test_output:
        parts = line.split()
        if len(parts) > 0 and parts[0] == 'all':
            try:
                float_vals = []
                for p in parts:
                    try:
                        float_vals.append(float(p))
                    except ValueError:
                        continue
                if len(float_vals) >= 4:
                    test_p     = float_vals[-4]
                    test_r     = float_vals[-3]
                    test_map50 = float_vals[-2]
                    test_map95 = float_vals[-1]
            except Exception:
                pass

    print("\n" + "━" * 50)
    print("📈 YOLOv7 TEST METRICS")
    print("━" * 50)
    print(f"Precision:   {test_p:.4f}")
    print(f"Recall:      {test_r:.4f}")
    print(f"mAP@50:      {test_map50:.4f}")
    print(f"mAP@50-95:   {test_map95:.4f}")
    print("━" * 50)
    print(f"📂 Results saved in: {TEST_OUTPUT_DIR}")


✅ Loading best model: /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt
🚀 Starting Testing…
Namespace(weights=['/teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt'], data='/teamspace/studios/this_studio/Volleyball01/YOLOv7/volleyball-2/data.yaml', batch_size=4, img_size=640, conf_thres=0.001, iou_thres=0.65, task='test', device='0', single_cls=False, augment=False, verbose=True, save_txt=True, save_hybrid=False, save_conf=True, save_json=True, project='/teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/val', name='volleyball_test', exist_ok=True, no_trace=False, v5_metric=False)
YOLOR 🚀 v0.1-128-ga207844 torch 2.5.1+cu121 CUDA:0 (Tesla T4, 14911.6875MB)

/teamspace/studios/this_studio/Volleyball01/YOLOv7/yolov7/models/experimental.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to const

## 10. Extract Training Metrics from results.txt

In [14]:
# YOLOv7 saves per-epoch metrics to results.txt
results_file = os.path.join(TRAIN_OUTPUT_DIR, 'results.txt')
train_p = train_r = train_map50 = train_map95 = 0.0

if os.path.isfile(results_file):
    with open(results_file, 'r') as f:
        lines = [l.strip() for l in f.readlines() if l.strip()]
    
    if lines:
        # Use the last numeric line
        last_line = lines[-1]
        parts = last_line.split()
        if len(parts) >= 12:
            try:
                # In results.txt, indices 8-11 are P, R, mAP@.5, mAP@.5:.95
                # We skip parts[0] because it's like '99/99'
                train_p     = float(parts[8])
                train_r     = float(parts[9])
                train_map50 = float(parts[10])
                train_map95 = float(parts[11])
            except Exception as e:
                print(f'⚠️ Error parsing results.txt: {e}')

    print('━' * 50)
    print('📈 YOLOv7 TRAINING FINAL METRICS (last epoch)')
    print('━' * 50)
    print(f'Precision:   {train_p:.4f}')
    print(f'Recall:      {train_r:.4f}')
    print(f'mAP@50:      {train_map50:.4f}')
    print(f'mAP@50-95:   {train_map95:.4f}')
    print('━' * 50)
else:
    print(f'❌ results.txt not found at: {results_file}')


⚠️ Could not parse results.txt: could not convert string to float: '99/99'
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📈 YOLOv7 TRAINING FINAL METRICS (last epoch)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Precision:   0.0000
Recall:      0.0000
mAP@50:      0.0000
mAP@50-95:   0.0000
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━


## 11. Save All Metrics to CSV

In [15]:
CSV_PATH = os.path.join(SCRIPT_DIR, "metrics_summary.csv")

rows = [
    ["Phase", "Precision", "Recall", "mAP50", "mAP50-95"],
    ["Train",  f"{train_p:.4f}",   f"{train_r:.4f}",   f"{train_map50:.4f}",  f"{train_map95:.4f}"],
    ["Valid",  f"{val_p:.4f}",     f"{val_r:.4f}",     f"{val_map50:.4f}",    f"{val_map95:.4f}"],
    ["Test",   f"{test_p:.4f}",    f"{test_r:.4f}",    f"{test_map50:.4f}",   f"{test_map95:.4f}"],
]

with open(CSV_PATH, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(rows)

print(f"✅ Metrics saved to: {CSV_PATH}")
print()

# Pretty-print the table
print(f"{'Phase':<8} {'Precision':>10} {'Recall':>10} {'mAP50':>10} {'mAP50-95':>10}")
print("─" * 52)
for row in rows[1:]:
    print(f"{row[0]:<8} {row[1]:>10} {row[2]:>10} {row[3]:>10} {row[4]:>10}")

✅ Metrics saved to: /teamspace/studios/this_studio/Volleyball01/YOLOv7/metrics_summary.csv

Phase     Precision     Recall      mAP50   mAP50-95
────────────────────────────────────────────────────
Train        0.0000     0.0000     0.0000     0.0000
Valid        0.6250     0.3330     0.3230     0.1430
Test         1.0000     0.2330     0.3350     0.1840


In [16]:
# ── Final Summary ──
print("\n" + "=" * 60)
print("  YOLOv7 VOLLEYBALL PIPELINE COMPLETE")
print("=" * 60)
print(f"  📂 Training results   : {TRAIN_OUTPUT_DIR}")
print(f"  📂 Validation results : {VAL_OUTPUT_DIR}")
print(f"  📂 Test results       : {TEST_OUTPUT_DIR}")
print(f"  📊 Metrics CSV        : {CSV_PATH}")
print(f"  📝 Training log       : {LOG_FILE}")
print(f"  🏆 Best weights       : {BEST_WEIGHTS}")
print("=" * 60)


  YOLOv7 VOLLEYBALL PIPELINE COMPLETE
  📂 Training results   : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train
  📂 Validation results : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/val/volleyball_val
  📂 Test results       : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/val/volleyball_test
  📊 Metrics CSV        : /teamspace/studios/this_studio/Volleyball01/YOLOv7/metrics_summary.csv
  📝 Training log       : /teamspace/studios/this_studio/Volleyball01/YOLOv7/logs/training_20260329_100730.log
  🏆 Best weights       : /teamspace/studios/this_studio/Volleyball01/YOLOv7/runs/volleyball_train/weights/best.pt
